<a href="https://colab.research.google.com/github/Srinithi-A/SRIML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinithi-A/SRIML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 1. My Rule and its Reason Codes

## Baseline Rule

The baseline rule identifies webpages that are likely to benefit from optimization based on historical search performance.

A page receives a higher score if it has:
- Low click-through rate (CTR)
- High search impressions
- Poor average search position
- Older content

The higher the score, the higher the priority for review.

## Reason Codes

- LOW_CTR – The page has a low click-through rate.
- HIGH_IMPRESSIONS – The page receives many impressions but relatively few clicks.
- LOW_POSITION – The average search position is poor.
- STALE_CONTENT – The content is relatively old and may need updating.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [7]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/Srinithi-A/SRIML/refs/heads/main/data/raw/content_refresh_anonymized.csv")

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [8]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [9]:
df["baseline_score"] = (
    (df["ctr"] < 0.05).astype(int)
    + (df["impressions_90d"] > 100).astype(int)
    + (df["avg_position"] > 20).astype(int)
    + (df["content_age_days"] > 365).astype(int)
)

In [10]:
def get_reason(row):

    reasons = []

    if row["ctr"] < 0.05:
        reasons.append("LOW_CTR")

    if row["impressions_90d"] > 100:
        reasons.append("HIGH_IMPRESSIONS")

    if row["avg_position"] > 20:
        reasons.append("LOW_POSITION")

    if row["content_age_days"] > 365:
        reasons.append("STALE_CONTENT")

    return ", ".join(reasons)

df["reason_code"] = df.apply(get_reason, axis=1)

In [11]:
def get_action(score):

    if score >= 3:
        return "Optimize Now"

    elif score == 2:
        return "Review"

    else:
        return "Monitor"

df["action"] = df["baseline_score"].apply(get_action)

In [12]:
queue = df.sort_values(
    by="baseline_score",
    ascending=False
)

queue.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
1191,content_3bac7b908a99,client_e629fa6598,10.0,0.05,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.00,0.00,0.0,moderate,page_3_5,down,-96.9,4,"LOW_CTR, HIGH_IMPRESSIONS, LOW_POSITION, STALE...",Optimize Now
23010,content_85aa5489f38b,client_19581e27de,880.0,0.67,HIGH,0.34,keyword article,commercial,NaN,NaN,...,0.00,0.00,0.0,low,page_3_5,up,36.0,4,"LOW_CTR, HIGH_IMPRESSIONS, LOW_POSITION, STALE...",Optimize Now
23017,content_c4e97dde2f71,client_2c624232cd,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,50.00,0.0,moderate,deep,up,141.0,4,"LOW_CTR, HIGH_IMPRESSIONS, LOW_POSITION, STALE...",Optimize Now
23032,content_dd6319753667,client_19581e27de,30.0,0.06,LOW,4.57,keyword article,informational,NaN,NaN,...,9.30,11.63,0.0,moderate,page_3_5,stable,-16.9,4,"LOW_CTR, HIGH_IMPRESSIONS, LOW_POSITION, STALE...",Optimize Now
2411,content_adeed5c1fd73,client_19581e27de,1000.0,0.95,HIGH,1.62,keyword article,transactional,NaN,NaN,...,0.00,0.00,0.0,moderate,deep,up,95.2,4,"LOW_CTR, HIGH_IMPRESSIONS, LOW_POSITION, STALE...",Optimize Now
12885,content_a145d3a84a67,client_e629fa6598,10.0,0.29,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.00,0.00,0.0,moderate,deep,up,166.7,4,"LOW_CTR, HIGH_IMPRESSIONS, LOW_POSITION, STALE...",Optimize Now
23039,content_e178b9980a0f,client_19581e27de,260.0,1.00,HIGH,1.90,keyword article,commercial,NaN,NaN,...,0.00,0.00,0.0,moderate,page_3_5,up,63.2,4,"LOW_CTR, HIGH_IMPRESSIONS, LOW_POSITION, STALE...",Optimize Now
7001,content_71bee40f7f97,client_19581e27de,1600.0,0.01,LOW,0.57,keyword article,commercial,NaN,NaN,...,0.00,0.00,0.0,moderate,deep,up,30.2,4,"LOW_CTR, HIGH_IMPRESSIONS, LOW_POSITION, STALE...",Optimize Now
23056,content_2c83d636c2bf,client_19581e27de,10.0,0.06,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,0.00,0.0,low,deep,up,29.2,4,"LOW_CTR, HIGH_IMPRESSIONS, LOW_POSITION, STALE...",Optimize Now
28754,content_467db1d7f145,client_4e07408562,10.0,0.14,LOW,0.00,keyword article,informational,2650.0,16463.0,...,0.00,20.00,0.0,moderate,page_3_5,down,-58.2,4,"LOW_CTR, HIGH_IMPRESSIONS, LOW_POSITION, STALE...",Optimize Now


In [13]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

CSV saved successfully!


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


| Rank | Action       | Reason Code               | Confidence | What would make it wrong?               |
| ---- | ------------ | ------------------------- | ---------- | --------------------------------------- |
| 1    | Optimize Now | LOW_CTR, HIGH_IMPRESSIONS | High       | Seasonal traffic changes                |
| 2    | Optimize Now | LOW_POSITION              | Medium     | Recent content update not yet reflected |
| 3    | Review       | STALE_CONTENT             | Medium     | Content already refreshed recently      |
| 4    | Review       | LOW_CTR                   | Medium     | Limited historical data                 |
| 5    | Review       | HIGH_IMPRESSIONS          | Medium     | Temporary search trends                 |
| 6    | Monitor      | LOW_POSITION              | Low        | Position improving naturally            |
| 7    | Monitor      | STALE_CONTENT             | Low        | Evergreen content performing well       |
| 8    | Review       | LOW_CTR                   | Medium     | CTR affected by branded searches        |
| 9    | Optimize Now | HIGH_IMPRESSIONS          | High       | Impressions fluctuate seasonally        |
| 10   | Review       | LOW_POSITION              | Medium     | Recent SEO changes pending              |
| 11   | Monitor      | LOW_CTR                   | Low        | Small sample size                       |
| 12   | Review       | STALE_CONTENT             | Medium     | Page updated outside measured window    |
| 13   | Review       | HIGH_IMPRESSIONS          | Medium     | Temporary campaign effects              |
| 14   | Monitor      | LOW_POSITION              | Low        | Ranking stabilising                     |
| 15   | Optimize Now | LOW_CTR                   | High       | SERP layout changes                     |
| 16   | Review       | STALE_CONTENT             | Medium     | Historical lag in metrics               |
| 17   | Monitor      | HIGH_IMPRESSIONS          | Low        | Traffic spike was temporary             |
| 18   | Review       | LOW_POSITION              | Medium     | Competitive changes                     |
| 19   | Monitor      | LOW_CTR                   | Low        | Insufficient history                    |
| 20   | Review       | STALE_CONTENT             | Medium     | Content already scheduled for update    |


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Weak Picks

Some recommendations may be incorrect because the baseline rule only considers a small number of historical signals.

Pages affected by seasonality, recent content updates, or changing user intent may receive lower or higher scores than they deserve.

# Leakage Check

The baseline rule only uses information available before the decision is made.

No future performance metrics, product flags, or label-derived features were used when calculating the baseline score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.